## Giải thích cơ chế Recursive Chunking
 
**Recursive chunking** là phương pháp tách văn bản thành các đoạn nhỏ (chunk) dựa trên thứ tự ưu tiên các ký tự phân tách (separators).
- Thuật toán sẽ kiểm tra từng separator (ví dụ: xuống dòng kép, xuống dòng đơn, khoảng trắng, ký tự bất kỳ) để tách văn bản thành các đoạn có độ dài mong muốn.
- Nếu đoạn văn bản sau khi tách vẫn lớn hơn `chunk_size`, thuật toán sẽ tiếp tục tách đoạn đó bằng separator tiếp theo.
- Quá trình này lặp lại (đệ quy) cho đến khi tất cả các đoạn đều nhỏ hơn hoặc bằng `chunk_size`.
- Có thể thiết lập `chunk_overlap` để các đoạn có phần giao nhau, giúp giữ ngữ cảnh giữa các chunk.
 
**Ưu điểm:**
- Giữ được cấu trúc logic của văn bản (ưu tiên tách theo đoạn, câu, từ).
- Phù hợp cho các bài toán NLP, RAG, embedding khi cần chia nhỏ dữ liệu mà vẫn giữ ngữ cảnh.
 
**Ví dụ separator:** `["\n\n", "\n", " ", ""]` (ưu tiên tách theo đoạn, sau đó câu, từ, cuối cùng là ký tự bất kỳ).

In [4]:
docs  = """
MARLEY'S GHOST

Marley was dead, to begin with. There is no doubt whatever about that. The register of his burial was signed by the clergyman, the clerk, the undertaker, and the chief mourner. Scrooge signed it. And Scrooge's name was good upon 'Change for anything he chose to put his hand to. Old Marley was as dead as a door-nail.

Mind! I don't mean to say that I know of my own knowledge, what there is particularly dead about a door-nail. I might have been inclined, myself, to regard a coffin-nail as the deadest piece of ironmongery in the trade. But the wisdom of our ancestors is in the simile; and my unhallowed hands shall not disturb it, or the country's done for. You will, therefore, permit me to repeat, emphatically, that Marley was as dead as a door-nail.

Scrooge knew he was dead? Of course he did. How could it be otherwise? Scrooge and he were partners for I don't know how many years. Scrooge was his sole executor, his sole administrator, his sole assign, his sole particularly dead about a door-nail. I might have been inclined, myself, to regard a coffin-nail as the deadest piece of ironmongery in the trade. But the wisdom of our ancestors is in the simile; and my unhallowed hands shall not disturb it, or the country's done for. You will, therefore, permit me to repeat, emphatically, that Marley was as dead as a door-nail.



"""

### Quy trình hoạt động của RecursiveCharacterTextSplitter (LangChain)
 
Khi một chunk quá lớn (vượt quá `chunk_size`), thuật toán sẽ thử cắt nhỏ dần theo thứ tự ưu tiên các separator.
 
**Các bước hoạt động:**
1. **Tách bằng separator ưu tiên cao nhất** (thường là `\n\n` – đoạn văn):
    - Nếu các đoạn nhỏ hơn hoặc bằng `chunk_size` → dùng luôn.
    - Nếu vẫn có đoạn quá dài → sang bước tiếp theo.
2. **Tách bằng separator tiếp theo** (ví dụ `\n` – ngắt dòng):
    - Nếu thành công → dùng.
    - Nếu vẫn dài quá → sang bước tiếp theo.
3. **Tách theo câu** (dấu `.`, `?`, `!`):
    - Nếu thành công → dùng.
    - Nếu vẫn dài quá → sang bước tiếp theo.
4. **Tách theo từ** (khoảng trắng):
    - Nếu thành công → dùng.
    - Nếu vẫn dài quá → sang bước tiếp theo.
5. **Fallback:** Nếu vẫn không thể chia nhỏ vừa `chunk_size`, thuật toán sẽ cắt thô theo số ký tự (`""` – từng ký tự một).
 
**Lưu ý:** Quá trình này lặp lại (đệ quy) cho đến khi tất cả các đoạn đều nhỏ hơn hoặc bằng `chunk_size`.

In [8]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

recursive_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    separators=["\n\n", "\n", " ",""],# ưu tiên tách theo ký tự xuống dòng kép, rồi xuống dòng, rồi space, cuối cùng tách theo ký tự
    length_function=len, # hàm tính độ dài của văn bản
    is_separator_regex=False, # các ký tự trong separators không phải là regex
)
# Tách văn bản

chunks = recursive_text_splitter.split_text(docs)
# In kết quả
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: '{chunk}'\n---")
    print(f"Length: {len(chunk)}\n===") 

Chunk 1: 'MARLEY'S GHOST'
---
Length: 14
===
Chunk 2: 'Marley was dead, to begin with. There is no doubt whatever about that. The register of his burial'
---
Length: 97
===
Chunk 3: 'of his burial was signed by the clergyman, the clerk, the undertaker, and the chief mourner.'
---
Length: 92
===
Chunk 4: 'the chief mourner. Scrooge signed it. And Scrooge's name was good upon 'Change for anything he'
---
Length: 94
===
Chunk 5: 'for anything he chose to put his hand to. Old Marley was as dead as a door-nail.'
---
Length: 80
===
Chunk 6: 'Mind! I don't mean to say that I know of my own knowledge, what there is particularly dead about a'
---
Length: 98
===
Chunk 7: 'dead about a door-nail. I might have been inclined, myself, to regard a coffin-nail as the deadest'
---
Length: 98
===
Chunk 8: 'as the deadest piece of ironmongery in the trade. But the wisdom of our ancestors is in the simile;'
---
Length: 99
===
Chunk 9: 'is in the simile; and my unhallowed hands shall not disturb it, or the

Có thể thấy rằng tất cả các chunk đều nhỏ hơn giới hạn của chunk_size là 100

**Lưu ý:** Chỉ những chunk nào quá dài và được đệ quy cắt tiếp, ví dụ như `chunk 2`, `chunk 3`, `chunk 4` là được đệ quy cắt ra từ một chunk nên mới có overlap 

In [11]:
# Bạn có thể thử các chunk_size và chunk_overlap khác nhau để thấy sự khác biệt

recursive_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20,
    separators=["\n\n", "\n", " ",""],# ưu tiên tách theo ký tự xuống dòng kép, rồi xuống dòng, rồi space, cuối cùng tách theo ký tự
    length_function=len, # hàm tính độ dài của văn bản
    is_separator_regex=False, # các ký tự trong separators không phải là regex
    add_start_index=True # thêm chỉ số bắt đầu của chunk trong văn bản gốc (dùng để trace back vị trí chunk trong văn bản gốc nếu cần thiết)
)
# Tách văn bản

chunks = recursive_text_splitter.create_documents([docs])

# In kết quả
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: '{chunk.page_content}'\n---")
    print(f"Length: {len(chunk.page_content)}")
    print(f"Start Index in original doc: {chunk.metadata}\n===")

Chunk 1: 'MARLEY'S GHOST'
---
Length: 14
Start Index in original doc: {'start_index': 1}
===
Chunk 2: 'Marley was dead, to begin with. There is no doubt whatever about that. The register of his burial was signed by the clergyman, the clerk, the'
---
Length: 141
Start Index in original doc: {'start_index': 17}
===
Chunk 3: 'the clerk, the undertaker, and the chief mourner. Scrooge signed it. And Scrooge's name was good upon 'Change for anything he chose to put his hand'
---
Length: 147
Start Index in original doc: {'start_index': 144}
===
Chunk 4: 'to put his hand to. Old Marley was as dead as a door-nail.'
---
Length: 58
Start Index in original doc: {'start_index': 276}
===
Chunk 5: 'Mind! I don't mean to say that I know of my own knowledge, what there is particularly dead about a door-nail. I might have been inclined, myself, to'
---
Length: 148
Start Index in original doc: {'start_index': 336}
===
Chunk 6: 'myself, to regard a coffin-nail as the deadest piece of ironmongery in the tr

### Lưu ý khi chọn chunk_size và chunk_overlap cho `recursive embedding`
 
- Bạn có thể thử các giá trị `chunk_size` và `chunk_overlap` khác nhau để phù hợp với từng loại văn bản hoặc từng model embedding.
- Ví dụ với OpenAI text-embedding-ada-002 thì tối đa 8191 token, và để tối ưu cho embedding nên chọn `chunk_size` từ 200 - 400 token (tương đương 150–300 từ tiếng Anh, tiếng Việt ngắn hơn chút).
- Nếu gửi chunk quá dài cho model embedding, sẽ gặp các vấn đề:
    - **Dài dòng, tốn chi phí** (embedding vector lớn, xử lý chậm, tốn tiền).
    - **Nguy cơ semantic drift**: model phải “nén” quá nhiều ý vào một vector duy nhất, làm giảm độ tập trung và chất lượng embedding.
 
**Khuyến nghị:**
- Nên kiểm tra giới hạn token của model embedding bạn sử dụng.
- Chia chunk vừa phải để giữ ngữ cảnh, tối ưu chi phí và chất lượng embedding.